# Comparación de Modelos Predictivos para Distemper Canino (CDV)

Este notebook permite la exploración interactiva y comparación rigurosa entre las tres arquitecturas evaluadas:
1. **Árbol de Decisión (`DecisionTreeClassifier`)**
2. **Random Forest (`RandomForestClassifier`)**
3. **Regresión Logística (`LogisticRegression`)**

Protocolo de validación: **Validación Cruzada Estratificada (StratifiedKFold con 5 pliegues)** sobre el dataset oficial `dataset_moquillo_real_v5.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.dataValidation import CDVDataValidator
from src.models.DecisionTreeTraining import DecisionTreeModelTrainer
from src.models.RandomForest import RandomForestModelTrainer
from src.models.LogisticRegression import LogisticRegressionModelTrainer

# 1. Cargar y validar dataset limpio
df = pd.read_csv('../data/raw/dataset_moquillo_real_v5.csv')
validator = CDVDataValidator()
clean_df, report = validator.validate_and_clean(df)

X = clean_df[list(CDVDataValidator.FEATURE_COLUMNS)]
y = clean_df[CDVDataValidator.TARGET_COLUMN].astype(int)

print(f'Muestras cargadas: {len(clean_df)} (Positivos CDV: {y.sum()}, Sanos: {(y==0).sum()})')
print('Reporte de calidad:', report.to_dict())

In [ ]:
# 2. Entrenamiento y evaluación Out-Of-Fold (OOF)
trainers = {
    'DecisionTree': DecisionTreeModelTrainer(n_splits=5, random_state=42),
    'RandomForest': RandomForestModelTrainer(n_splits=5, random_state=42),
    'LogisticRegression': LogisticRegressionModelTrainer(n_splits=5, random_state=42)
}

results = {}
for name, trainer in trainers.items():
    print(f'Ajustando {name}...')
    trainer.fit(X, y)
    eval_res = trainer.evaluate_cv(X, y)
    results[name] = eval_res
    metrics = eval_res.aggregate_metrics
    print(f"{name} -> Recall: {metrics['recall']:.4f}, Especificidad: {metrics.get('specificity', 0.0):.4f}, F1: {metrics['f1']:.4f}, Accuracy: {metrics['accuracy']:.4f}")

In [ ]:
# 3. Visualización comparativa de métricas
metrics_df = pd.DataFrame([
    {
        'Modelo': name,
        'Sensibilidad (Recall)': res.aggregate_metrics['recall'],
        'Especificidad': res.aggregate_metrics.get('specificity', 0.0),
        'F1-Score': res.aggregate_metrics['f1'],
        'Exactitud (Accuracy)': res.aggregate_metrics['accuracy']
    }
    for name, res in results.items()
]).set_index('Modelo')

display(metrics_df)

fig, ax = plt.subplots(figsize=(10, 5))
metrics_df.plot(kind='bar', ax=ax, colormap='viridis', edgecolor='black')
ax.axhline(0.80, color='red', linestyle='--', label='Meta Clínica Recall >= 80%')
ax.set_title('Comparativa de Rendimiento Clínico Out-Of-Fold')
ax.set_ylabel('Valor de la Métrica [0.0 - 1.0]')
ax.set_ylim(0, 1.1)
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()